# 03 — Paint & Damage Feature Engineering (Team 3)

## Responsibility
This notebook handles the paint and bodywork damage columns. These are the most
complex columns in the dataset because they contain free-text lists of car body
parts rather than simple categorical values.

## Columns Assigned
`Boya-değişen`, `Orjinal`, `Lokal boyalı`, `Boyalı`, `Değişmiş`, `Belirtilmemiş`

## Understanding the Data
These 6 columns together describe the paint/body condition of the car.
Each cell may contain:
- A dash "-" meaning none/not applicable
- The text "Tamamı orjinal" meaning fully original
- A space-separated list of car body part names that have been painted/changed
  (e.g. "Sağ Arka Çamurluk Motor Kaputu Ön Tampon")

## What is Expected as Output
Do NOT label-encode these columns as text. Instead, engineer meaningful numeric features:
- `total_painted_parts` (int): total count of parts listed across Boyalı + Lokal boyalı
- `total_changed_parts` (int): count of parts listed in Değişmiş
- `is_fully_original` (int 0/1): 1 if Orjinal == "Tamamı orjinal"
- `paint_damage_score` (int): total number of body parts mentioned across ALL 6 columns
- `Boya-değişen_count` (int): number of parts listed in Boya-değişen (0 if "-")
- Drop all original 6 text columns after feature engineering
- No nulls remaining

## Input
raw_dataset.csv from GitHub raw URL

## Output
df_team3: a DataFrame slice containing only the engineered numeric columns above,
with the same index as the original dataset (3424 rows)

---

> ## ⚠️ READ THIS BEFORE YOU WRITE A SINGLE LINE OF CODE ⚠️
>
> ### The code below is a PLACEHOLDER — it is NOT the final solution.
>
> The cells in this notebook show **one possible way** to process the assigned columns. You are **not** required to follow this approach. You may completely replace it with your own method if you believe yours is better suited to the data.
>
> ### What IS required — without exception:
>
> **Every code cell must be accompanied by a markdown cell that explains:**
> 1. **What you did** — describe the operation in plain language.
> 2. **Why you did it** — justify the choice. Why did you think this specific approach would work? What problem does it solve? Why this method and not another?
>
> A notebook that only contains code — or markdown cells that just repeat the code in words — will **not** be accepted.
>
> **Not acceptable:** `## Step 2` or `# fill nulls`
>
> **Acceptable:** *"We imputed missing values in `Kilometre` with the median rather than the mean because the column is heavily right-skewed (a small number of very high-mileage listings would pull the mean upward, misrepresenting the majority of vehicles). The median is robust to those outliers and better reflects a typical listing."*
>
> Document every decision. If you tried an approach and abandoned it, explain why.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

RAW_URL = "https://raw.githubusercontent.com/MamoMGD1/ISE302-DataMining-GroupProject/main/data/raw_dataset.csv"
df_full = pd.read_csv(RAW_URL)
print(f"Loaded dataset: {df_full.shape}")
df_full.head(3)

In [ ]:
MY_COLUMNS = ['Boya-değişen', 'Orjinal', 'Lokal boyalı',
              'Boyalı', 'Değişmiş', 'Belirtilmemiş']
df = df_full[MY_COLUMNS].copy()

## Step 1 — Null Analysis

In [ ]:
# Print null counts and percentage for each column
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(2)
print(pd.DataFrame({'null_count': null_counts, 'null_pct': null_pct}))

In [ ]:
# Helper: count space-separated body parts in a cell; treat '-', 'Tamamı orjinal', and NaN as 0 parts
def count_parts(cell):
    if pd.isna(cell):
        return 0
    val = str(cell).strip()
    if val == '-' or val == '' or val.lower() == 'tamamı orjinal':
        return 0
    return len(val.split())

# Fill NaN values with '-' so all cells are strings before feature engineering
df = df.fillna('-')
print(f"Nulls after fill: {df.isnull().sum().sum()}")

In [ ]:
# Engineer total_painted_parts: total count of parts in Boyalı + Lokal boyalı
df['total_painted_parts'] = df['Boyalı'].apply(count_parts) + df['Lokal boyalı'].apply(count_parts)
print(f"total_painted_parts nulls: {df['total_painted_parts'].isnull().sum()}")

In [ ]:
# Engineer total_changed_parts: count of parts in Değişmiş
df['total_changed_parts'] = df['Değişmiş'].apply(count_parts)
print(f"total_changed_parts nulls: {df['total_changed_parts'].isnull().sum()}")

In [ ]:
# Engineer is_fully_original: 1 if Orjinal == 'Tamamı orjinal', else 0
df['is_fully_original'] = (df['Orjinal'].str.strip() == 'Tamamı orjinal').astype(int)
print(f"is_fully_original nulls: {df['is_fully_original'].isnull().sum()}")

In [ ]:
# Engineer paint_damage_score: total parts mentioned across ALL 6 columns
df['paint_damage_score'] = sum(df[col].apply(count_parts) for col in MY_COLUMNS)
print(f"paint_damage_score nulls: {df['paint_damage_score'].isnull().sum()}")

In [ ]:
# Engineer Boya-değişen_count: number of parts listed in Boya-değişen (0 if '-')
df['Boya-değişen_count'] = df['Boya-değişen'].apply(count_parts)
print(f"Boya-değişen_count nulls: {df['Boya-değişen_count'].isnull().sum()}")

In [ ]:
# Drop all original 6 text columns; keep only the engineered numeric features
df.drop(columns=MY_COLUMNS, inplace=True)
print(f"Shape after dropping original columns: {df.shape}")

In [ ]:
df_team3 = df.copy()
assert df_team3.isnull().sum().sum() == 0, "Nulls remain!"
assert df_team3.select_dtypes(exclude='number').shape[1] == 0, "Non-numeric columns remain!"
print(f"✅ Team 3 output ready. Shape: {df_team3.shape}")
print(df_team3.dtypes)